# FakeAVCeleb Audio Preprocessing (Kaggle)
This notebook preprocesses FakeAVCeleb audio into log-mel PNGs and optionally zips them for reuse.
Use split-by-split runs to avoid Kaggle session drops.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

DATASET_ROOT = '/kaggle/input/datasets/uzairfar00q/fakeavceleb/FakeAVCeleb'
OUT_DIR = '/kaggle/working/fakeav_audio_png'
SPLITS = ['train', 'validation', 'test']
MAX_VIDEOS_PER_CLASS = 200

# Toggle these before running
RUN_PREPROCESS = True
RUN_ZIP = False


In [ ]:
print('Dataset root exists:', os.path.exists(DATASET_ROOT))
print('Top-level folders:', os.listdir(DATASET_ROOT))

print('Summary-only check...')
subprocess.run([
    sys.executable, '-u',
    '/kaggle/working/swin-model/ForensiCore-fakeav-audio-preprocess.py',
    '--root', DATASET_ROOT,
    '--out', OUT_DIR,
    '--train-split', '0.8',
    '--val-split', '0.1',
    '--test-split', '0.1',
    '--summary-only',
], check=True)


In [ ]:
def count_pngs(root, split, label):
    path = Path(root) / split / label
    if not path.exists():
        return 0
    return sum(1 for _ in path.glob('*.png'))

for split in SPLITS:
    real_count = count_pngs(OUT_DIR, split, 'real')
    fake_count = count_pngs(OUT_DIR, split, 'fake')
    print(f'{split}: {real_count} real, {fake_count} fake')


In [ ]:
def run_preprocess(split_name, max_per_class):
    print(f'Preprocessing {split_name} (max_per_class={max_per_class})...')
    subprocess.run([
        sys.executable, '-u',
        '/kaggle/working/swin-model/ForensiCore-fakeav-audio-preprocess.py',
        '--root', DATASET_ROOT,
        '--out', OUT_DIR,
        '--train-split', '0.8',
        '--val-split', '0.1',
        '--test-split', '0.1',
        '--split', split_name,
        '--max-videos-per-class', str(max_per_class),
    ], check=True)

if RUN_PREPROCESS:
    for split in SPLITS:
        run_preprocess(split, MAX_VIDEOS_PER_CLASS)


In [ ]:
def dir_size_bytes(root):
    total = 0
    for base, _, files in os.walk(root):
        for name in files:
            total += os.path.getsize(os.path.join(base, name))
    return total

if RUN_ZIP:
    size_bytes = dir_size_bytes(OUT_DIR)
    print(f'Current size: {size_bytes / (1024**2):.2f} MB')
    subprocess.run([
        'zip', '-r', '/kaggle/output/fakeav_audio_png.zip', OUT_DIR
    ], check=True)
    print('Saved zip to /kaggle/output/fakeav_audio_png.zip')
